# Agentic RAG and tool calling 

In [2]:
# We'll use a typical openAI sdk, so lets import that first

from openai import OpenAI

openai_client = OpenAI()

### Documents loading

In [3]:
# We'll have to bring our documents to use in this example from gitsource

from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

# we can parse and chunk straight from this package

### Create search index

In [8]:
from minsearch import AppendableIndex


In [7]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

### Traditional RAG

In [9]:
# Lets create our traditional rag, as we previously already created

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

# Now lets add in the rag prompt template

import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [10]:
# Create a prompt to add to the RAG

question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [11]:
# Grab the responses from the RAG

messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [12]:
# Get the answer from the RAG

print(response.output_text)

# We should now expect a bad response from the RAG because it does not
# handle typos well, its simple.

The provided context does not specify how to create a dashboard in Evidently. It mainly discusses setting up an account, connecting from Python, and features of the Evidently Platform, but does not include detailed steps for creating a dashboard.


### Making It Agentic
We'll convert this to an agentic approach where the LLM chooses whether to search and what query to use.

In [13]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

# The tool definition tells the LLM what the function does, what parameters it accepts, and which parameters are required.

# First, let's see what happens without tools - just the instructions and the question:

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)
response.usage.input_tokens


63

In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

# The input tokens increased from 63 to 110 because the tool description is now included in the system prompt.



110

## Handling the Tool Call
The LLM chose to call our search function. Let's inspect the tool call:

In [16]:
tool_call = response.output[0]
tool_call

# The LLM fixed the typo from "dahsbord" to "create dashboard in Evidently". Agentic RAG can reformulate queries.

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_seieWRYMD5c39qdNHwtiocBI', name='search', type='function_call', id='fc_0d561f6292a28b8e006afb04c64bd087d2baba3d414ed8d709', async_=None, caller=None, namespace=None, status='completed')

In [17]:
# Now append the tool call to our message history:
messages.append(tool_call)

In [18]:
# The response includes the arguments the LLM wants to pass:
tool_call.arguments


'{"query":"create dashboard in Evidently"}'

In [19]:
# Parse the arguments:

arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidently'}

In [20]:
# We execute the search with those arguments. First, let's try with an explicit query parameter:

search_results = search(query='create dashboard in Evidently')


In [21]:
# Alternatively, we can unpack the arguments dictionary:

search_results = search(**arguments)
search_results[:1]

# The output shows one search result with dashboard information.

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

## Continuing the Conversation

In [22]:
# Now format the search results as JSON and create the function call output:

call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [23]:
# Append the output to messages:

messages.append(call_output)
messages


[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_seieWRYMD5c39qdNHwtiocBI', name='search', type='function_call', id='fc_0d561f6292a28b8e006afb04c64bd087d2baba3d414ed8d709', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_seieWRYMD5c39qdNHwtiocBI',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation 

## Getting the Final Response

In [ ]:
# Now we can make another API call with the updated message history:

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

# Tokens: 3685. 
# The input tokens jumped to 4075 because we're sending the entire conversation history 
# including the search results.



3685

In [25]:
# The LLM now has the search results and can generate an informed answer:

print(response.output_text)

To create a dashboard in Evidently, follow these steps:

1. **Create a Project**: Start by creating a project in Evidently. You need at least one report saved in this project to populate your dashboard.

2. **Access the Dashboard**: Each project has its own dashboard, which will be empty initially.

3. **Add Panels**:
   - You can add panels using the user interface or via the Python API.
   - **Adding via UI** (Cloud and Enterprise only): Navigate through the dashboard interface to add panels directly.
   - **Adding via API**:
     - Import the necessary modules:
       ```python
       from evidently.sdk.models import PanelMetric
       from evidently.sdk.panels import DashboardPanelPlot
       ```
     - Use the following code template to add a panel:
       ```python
       project.dashboard.add_panel(
           DashboardPanelPlot(
               title="Your Panel Title",
               size="full",
               values=[PanelMetric(legend="Metric Name", metric="MetricType")],
  

## Structured Output
We can also get structured output from our agent. Let's define a response model:


In [26]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")


In [ ]:
# Get a structured response:

response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

# Tokens increased a bit because we're parsing the response into a structured format.

# Print the answer:

3909

In [28]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, follow these steps:

### 1. **Set Up Your Project**  
   - Create an **Organization** in Evidently, and then create a **Project** within that organization. This sets up the framework for your dashboard.

### 2. **Run an Evaluation**  
   - You need to run at least one evaluation and save a Report to your Project. The dashboard pulls data from these Reports.

### 3. **Add Panels**  
   - **Using UI (Cloud and Enterprise Versions)**: You can add panels directly from the dashboard interface. Each panel displays specific metrics. You can organize them into different tabs for a cleaner view.  
   - **Using Python API**: To create or add panels programmatically, import the necessary classes as follows:
     ```python
     from evidently.sdk.models import PanelMetric
     from evidently.sdk.panels import DashboardPanelPlot
     ```  
     - To add a panel, use:
     ```python
     project.dashboard.add_panel(
         DashboardPanelPlot(
             title=

## Agentic RAG vs Traditional RAG

The key difference:

Traditional RAG: - Always searches - Always includes results in prompt - Fixed flow: search, retrieve, generate - Can't handle typos or reformulate queries

Agentic RAG: - LLM chooses whether to search - LLM chooses the search query - Can fix typos and reformulate queries - Can use multiple tools in sequence - Continues until it has enough information

This makes agentic RAG more flexible. The LLM can: - Skip search if it already knows the answer - Reformulate the query if needed - Use multiple tools together - Reason about when it has enough information

The tradeoff is cost - each round trip adds tokens to the conversation. You saw the progression: 63 → 110 → 4075 → 4299.

